In [1]:
import duckdb
import pandas as pd
from pathlib import Path
import gc
import yaml
import logging
from collections import defaultdict, deque
from typing import Any
import re

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.expand_frame_repr', True)

logging.basicConfig(
    level=logging.DEBUG,
    format="%(asctime)s | %(levelname)s | %(message)s",
    force=True 
)


In [2]:
class ConnectionManager():
    def __init__(self, db_con_str):
        # la fonction duckdb.connect transforme le path en absolue, autant le faire ici 
        self.db_con_str = Path(db_con_str).expanduser().resolve()

        # connexion lazy
        self._con = None


    @property
    def con(self):
        if(self._con is None):
            # se connecter à la base de données
            self._con = duckdb.connect(self.db_con_str)

        return self._con


    @con.setter
    def con(self, value):
        # si au moment de changer la connexion on a déjà une connexion active
        if(self._con is not None):
            self._con.close() # cloturer la connexion en cours
            self._con = None # retirer la référence sur la connexion en cours
            gc.collect() # appeler le garbage collector pour forcer l'action de libérer les ressources et éviter les conflits d'accès

        self._con = value # pointer sur la nouvelle connexion
    
    
    def close_con(self):
        # on exploite le setter de la propriété pour cloturer correctement la connexion
        self.con = None


    def __del__(self):
        """Ferme automatiquement la connexion DuckDB quand l'objet est détruit."""
        try:
            self.close_con()
        except Exception:
            pass


    def __enter__(self):
        return self
    

    def __exit__(self, exc_type, exc_val, exc_tb):
        self.close_con()
        return False

In [3]:
class ConnectionUtils(ConnectionManager):
    def __init__(self, db_con_str : str):
        super().__init__(db_con_str)


    def tables(self):
        """Retourne la liste de toutes les tables physiques de la base courante"""
        return self.con.sql("""
            SELECT table_name
            FROM information_schema.tables
            WHERE table_catalog = current_database()
            AND table_type = 'BASE TABLE'
            ORDER BY table_name
        """)


    def views(self):
        """Cette fonction renvoi la liste de toutes vues accessibles dans la base de données courante"""
        return self.con.sql("""
            SELECT table_name
            FROM information_schema.views
            WHERE table_catalog = current_database()
            ORDER BY table_name
        """)


    def tables_views(self):
        """Retourne la liste des tables et des vues de la base courante"""
        return self.con.sql("""
            SELECT 
                table_name,
                table_type          -- 'BASE TABLE' ou 'VIEW'
            FROM information_schema.tables
            WHERE table_catalog = current_database()
            ORDER BY table_type, table_name
        """)


    def table_exists(self, table_name : str):
        """Cette foction vérife qu'une table physique existe dans la base courante"""
        return self.con.sql(f"""
            SELECT COUNT(*) > 0 AS existe
            FROM information_schema.tables 
            WHERE 
                (table_name = '{table_name}')
                AND
                (table_type = 'BASE TABLE')
                AND
                (table_catalog = current_database())
        """).fetchone()[0]
    
    
    def view_exists(self, view_name : str):
        """Cette fonction check si une vue existe"""
        return self.con.sql(f"""
            SELECT COUNT(*) > 0 AS existe
            FROM information_schema.views 
            WHERE 
                (table_name = '{view_name}')
                AND
                (table_catalog = current_database())
        """).fetchone()[0]


    def table_view_exists(self, name : str):
        """Cette foction vérife qu'une table physique ou une vue logique existe dans la base courante"""
        return self.con.sql(f"""
            SELECT COUNT(*) > 0 AS existe
            FROM information_schema.tables 
            WHERE 
                (table_name = '{name}')
                AND
                (table_type = 'BASE TABLE' OR table_type = 'VIEW')
                AND
                (table_catalog = current_database())
        """).fetchone()[0]


    def drop_table_if_exists(self, table_name : str):
        """Cette fonction permet de supprimer une table s'elle existe"""
        self.con.sql(f"DROP TABLE IF EXISTS {table_name}")


    def drop_tables_if_exists(self, tables : list[str]):
        """Cette fonction surpprime chaque table de la liste tables s'elle existe dans la base courante"""
        for table_name in tables : 
            self.drop_table_if_exists(table_name)


    def drop_view_if_exists(self, view_name : str):
        """Cette fonction permet de supprimer une vue s'elle existe"""
        self.con.sql(f"DROP VIEW IF EXISTS {view_name}")


    def drop_views_if_exists(self, views : list[str]):
        """Cette fonction surpprime chaque vue de la liste views s'elle existe dans la base courante"""
        for view_name in views : 
            self.drop_view_if_exists(view_name)


    def table(self, table_name : str):
        """Cette fonction renvoi la table dont le nom est passé en paramètre"""
        return self.con.table(table_name)


    def view(self, view_name : str):
        """Cette fonction renvoi la vue dont le nom est passé en paramètre"""
        return self.con.view(view_name)


    def table_view(self, name : str):
        """Retourne la relation d'une table ou d'une vue selon ce qui existe."""
        return self.con.sql(f"SELECT * FROM {name}")


    def create_table_view_if_not_exists(self, name: str, sql: str, type: str = "VIEW"):
        """
        Crée une vue ou une table uniquement si elle n'existe pas encore.
        """
        sql = sql.strip().rstrip(";")
        self.con.sql(f"""CREATE {type} IF NOT EXISTS {name} AS ({sql})""")
        

In [4]:
class DependencyTree:
    def __init__(self, data: dict[str, dict[str, Any]]):
        """
        data : dictionnaire de la forme
        {
            "v_sales": {"requires": ["t_sales"], ...},
            "t_sales": {"requires": ["df_sales"], ...},
            ...
        }
        """
        self.data = data
        self.graph = self._build_graph()          # node -> list of dependencies
        self.reverse_graph = self._build_reverse_graph()  # node -> list of dependents

    def _build_graph(self) -> dict[str, list[str]]:
        return {
            name: config.get("requires", [])
            for name, config in self.data.items()
        }

    def _build_reverse_graph(self) -> dict[str, list[str]]:
        reverse = defaultdict(list)
        for node, deps in self.graph.items():
            for dep in deps:
                reverse[dep].append(node)
        return dict(reverse)

    # -------------------------------------------------------------------------
    # Informations de base
    # -------------------------------------------------------------------------
    def nodes(self) -> list[str]:
        """Retourne tous les nœuds du graphe."""
        return list(self.graph.keys())

    def dependencies(self, name: str) -> list[str]:
        """Retourne les dépendances directes d'un nœud."""
        return self.graph.get(name, [])

    def dependents(self, name: str) -> list[str]:
        """Retourne les nœuds qui dépendent directement de celui-ci."""
        return self.reverse_graph.get(name, [])

    def roots(self) -> list[str]:
        """Nœuds qui ne sont requis par personne."""
        all_deps = {dep for deps in self.graph.values() for dep in deps}
        return [n for n in self.graph if n not in all_deps]

    def leaves(self) -> list[str]:
        """Nœuds qui n'ont aucune dépendance."""
        return [n for n, deps in self.graph.items() if not deps]

    # -------------------------------------------------------------------------
    # Dépendances récursives
    # -------------------------------------------------------------------------
    def all_dependencies(self, name: str) -> list[str]:
        """Retourne toutes les dépendances (directes + indirectes) dans l'ordre topologique."""
        result = []
        visited = set()

        def dfs(node: str):
            if node in visited:
                return
            visited.add(node)
            for dep in self.graph.get(node, []):
                dfs(dep)
            result.append(node)

        dfs(name)
        return result[:-1]  # on retire le nœud lui-même

    def creation_order(self, name: str | None = None) -> list[str]:
        """
        Ordre de création (topologique).
        Si name est fourni → uniquement pour ce nœud et ses dépendances.
        Sinon → ordre global.
        """
        if name:
            nodes = self.all_dependencies(name) + [name]
        else:
            nodes = self.nodes()

        in_degree = {n: 0 for n in nodes}
        for n in nodes:
            for dep in self.graph.get(n, []):
                if dep in in_degree:
                    in_degree[n] += 1

        queue = deque([n for n, deg in in_degree.items() if deg == 0])
        order = []

        while queue:
            node = queue.popleft()
            order.append(node)
            for dependent in self.reverse_graph.get(node, []):
                if dependent in in_degree:
                    in_degree[dependent] -= 1
                    if in_degree[dependent] == 0:
                        queue.append(dependent)

        return order

    # -------------------------------------------------------------------------
    # Affichage
    # -------------------------------------------------------------------------
    def print_tree(self, root: str | None = None):
        """Affiche l'arbre de dépendances en texte."""
        def _print(node: str, prefix: str = "", is_last: bool = True, visited: set | None = None):
            if visited is None:
                visited = set()

            connector = "└── " if is_last else "├── "
            print(f"{prefix}{connector}{node}")

            if node in visited:
                print(f"{prefix}{'    ' if is_last else '│   '}└── [cycle détecté]")
                return

            visited = visited | {node}
            deps = self.graph.get(node, [])
            new_prefix = prefix + ("    " if is_last else "│   ")

            for i, dep in enumerate(deps):
                _print(dep, new_prefix, i == len(deps) - 1, visited)

        if root:
            print(f"\n=== Dépendances de '{root}' ===\n")
            _print(root)
        else:
            print("\n=== Graphe complet ===\n")
            roots = self.roots()
            for i, r in enumerate(roots):
                _print(r, is_last=(i == len(roots) - 1))

    def print_levels(self, name: str | None = None):
        """Affiche les nœuds par niveau topologique."""
        order = self.creation_order(name)
        print(f"\n=== Ordre de création {'de ' + name if name else 'global'} ===\n")
        for i, node in enumerate(order, 1):
            print(f"{i:2d}. {node}")

In [5]:
class ConnectionPipeline(ConnectionUtils):
    def __init__(self, db_con_str : str, pipeline_file_path : str):
        super().__init__(db_con_str)
        self.pipeline_file_path = Path(pipeline_file_path).expanduser().resolve()
        self._pipeline = None
        self._tree = None


    def load_pipeline(self) -> dict:
        """Charge le fichier de définition des tables/vues."""
        with open(self.pipeline_file_path, "r", encoding="utf-8") as f:
            pipeline = yaml.safe_load(f)
            return pipeline


    @property
    def pipeline(self):
        if(self._pipeline is None):
            self._pipeline = self.load_pipeline()
        return self._pipeline


    @property
    def tree(self):
        if(self._tree is None):
            self._tree = DependencyTree(self.pipeline)
        return self._tree
    

    def df_from_file(self, file: str | Path, **kwargs) -> pd.DataFrame:
        """Charge un fichier en DataFrame selon son extension + options"""
        path = Path(file).expanduser().resolve()
        suffix = path.suffix.lower()

        if suffix in {".xlsx", ".xls", ".xlsm"}:
            return pd.read_excel(path, **kwargs)

        elif suffix == ".csv":
            return pd.read_csv(path, **kwargs)

        elif suffix == ".tsv":
            return pd.read_csv(path, sep="\t", **kwargs)

        elif suffix == ".json":
            return pd.read_json(path, **kwargs)

        elif suffix == ".parquet":
            return pd.read_parquet(path, **kwargs)

        else:
            raise ValueError(f"Extension non supportée : {suffix}")


    def df_from_file_config(self, config : dict):
        # On prépare les kwargs en enlevant les clés réservées
        reserved = {"type", "requires", "file"}
        kwargs = {k: v for k, v in config.items() if k not in reserved}

        return self.df_from_file(config["file"], **kwargs)


    def process_dataframe_type(self, name : str):
        if(name in self.pipeline):
            if(not self.table_view_exists(name)):
                config = self.pipeline[name]
                df = self.df_from_file_config(config)
                self.con.register(name, df)


    def process_table_view_type(self, name : str):
        if(name in self.pipeline):
            config = self.pipeline[name]
            self.create_table_view_if_not_exists(name, config["sql"], config["type"])


    def process(self, name : str):
        logging.getLogger().debug(f"process({name})")

        if(name in self.pipeline):
            config = self.pipeline[name]

            if(config["type"] == "dataframe"):
                self.process_dataframe_type(name)

            elif(config["type"] in ["table", "view"]):
                self.process_table_view_type(name)


    def process_with_requires(self, name : str):
        if(name in self.pipeline):
            config = self.pipeline[name]

            if(not self.table_view_exists(name)):
                for subname in config.get("requires", []):
                    self.process_with_requires(subname)

                self.process(name)


    def p_table_view(self, name : str):
        self.process_with_requires(name)
        return self.table_view(name)

In [6]:
class SalesPilBase(ConnectionPipeline):
    def __init__(self, db_con_str = "duckdb/pilotes/base/base.duckdb", pipeline_file_path = "config/base_pipeline.yaml"):
        super().__init__(db_con_str, pipeline_file_path)

    def sanitize(self, name: str) -> str:
        """Nettoie un nom pour en faire un préfixe de colonne valide."""
        name = str(name).strip().lower()
        name = re.sub(r"[^a-z0-9]+", "_", name)
        name = re.sub(r"_+", "_", name).strip("_")
        return name


    def pivot_sales_model_base(self, df: pd.DataFrame) -> pd.DataFrame:
        """
        Transforme v_sales_model_base (grain produit) en une table
        à **une ligne par hôtel et par scénario** avec colonnes préfixées :
        - produit__*
        - gamme__*
        - type__*
        - global__*

        Si ``scenario_id`` (ou ``scenario_label``) est présent, le groupby
        se fait sur (scenario_id, hotel_code) — indispensable pour les
        simulations de retrait (plusieurs mix produits pour le même hôtel).
        Sans colonne scénario : 1 ligne / hôtel (comportement historique).
        """

        # -------------------------------------------------
        # 1. Colonnes d'identité (hôtel + scénario)
        # -------------------------------------------------
        scenario_cols = [
            c for c in (
                "scenario_id",
                "scenario_label",
                "scenario_kind",
                "scenario_rank",
                "n_removed",
                "removed_items",
            )
            if c in df.columns
        ]
        id_cols = scenario_cols + [
            c for c in ("hotel_code", "hotel_name", "solution", "metres_lineaires")
            if c in df.columns
        ]

        # -------------------------------------------------
        # 2. Métriques par niveau
        # -------------------------------------------------
        product_metrics = [
            c for c in df.columns
            if c.startswith("produit_") and c not in id_cols
        ]
        gamme_metrics = [
            c for c in df.columns
            if c.startswith("gamme_") and c not in id_cols
        ]
        type_metrics = [
            c for c in df.columns
            if c.startswith("type_") and c not in id_cols
        ]
        global_metrics = [
            c for c in df.columns
            if not c.startswith(("produit_", "gamme_", "type_"))
            and c not in id_cols
            and c not in ["type", "gamme", "produit", "nombre_mois"]
            and c not in scenario_cols
        ]

        # Groupby : scénario × hôtel si scénario présent, sinon hôtel seul
        if "scenario_id" in df.columns:
            group_keys = ["scenario_id", "hotel_code"]
        elif "scenario_label" in df.columns:
            group_keys = ["scenario_label", "hotel_code"]
        else:
            group_keys = ["hotel_code"]

        rows = []
        for _, g in df.groupby(group_keys, sort=False, dropna=False):
            row = {}
            first = g.iloc[0]
            for col in id_cols:
                row[col] = first[col]

            for _, r in g.iterrows():
                prefix = f"produit__{self.sanitize(r['produit'])}__"
                for m in product_metrics:
                    row[prefix + m] = r[m]

            gammes = g.drop_duplicates(subset=["type", "gamme"] if "type" in g.columns else ["gamme"])
            for _, r in gammes.iterrows():
                prefix = f"gamme__{self.sanitize(r['gamme'])}__"
                for m in gamme_metrics:
                    row[prefix + m] = r[m]

            if "type" in g.columns:
                types = g.drop_duplicates(subset=["type"])
                for _, r in types.iterrows():
                    prefix = f"type__{self.sanitize(r['type'])}__"
                    for m in type_metrics:
                        row[prefix + m] = r[m]

            for m in global_metrics:
                row[f"global__{m}"] = first[m]

            rows.append(row)

        result = pd.DataFrame(rows)
        result = result.fillna(0)
        return result


    def create_or_replace_sales_model_base_line_table(self, source: str = "v_sales_model_base"):
        """
        Pivot large → t_sales_model_base_line.
        ``source`` : vue/table grain produit (éventuellement multi-scénarios).
        """
        df = self.pivot_sales_model_base(self.p_table_view(source).df())
        self.con.register("v_sales_model_base_line", df)
        self.con.sql(
            "CREATE OR REPLACE TABLE t_sales_model_base_line AS "
            "SELECT * FROM v_sales_model_base_line"
        )
        return df


    @classmethod
    def main(cls):
        """Cette fonction représente la fonction principale qui exploite cette classe et qu'elle faut exécuter"""

        base = SalesPilBase()

        print(base.p_table_view("v_sales").df().shape)
        display(base.p_table_view("v_sales").df().head(3))

        print(base.p_table_view("v_sales_model").df().shape)
        display(base.p_table_view("v_sales_model").df().head(3))

        print(base.p_table_view("v_sales_model_base").df().shape)
        display(base.p_table_view("v_sales_model_base").df().head(3))

        base.create_or_replace_sales_model_base_line_table()

        line = base.p_table_view("t_sales_model_base_line").df()
        print(line.shape, "# attendu base : 1 ligne / hôtel")
        display(line.head(3))

        base.close_con()


In [ ]:
SalesPilBase.main() # exécuter la fonction principale de la classe permettant de construire la base de données de modélisation

In [ ]:
class SalesPilSim(SalesPilBase):
    """
    Simulations de retrait (produits / gammes / types).

    Chaque itération de retrait est un **scénario** :
      - insert dans ``t_sales_model_sim`` avec scenario_id / scenario_label
      - grain produit (plusieurs lignes / hôtel / scénario)
    À la fin, pivot → ``t_sales_model_base_line`` :
      **1 ligne par (hôtel × scénario)**.
    """
    def __init__(self, db_con_str = "duckdb/pilotes/sim/sim.duckdb", pipeline_file_path = "config/sim_pipeline.yaml"):
        super().__init__(db_con_str, pipeline_file_path)
        self._scenario_seq = 0


    def remove_elements(self, champs : str, elements : list[str]):
        """Recalcule les indicateurs après retrait d'éléments (sans INSERT)."""

        list_str = ",".join([f"'{elm}'" for elm in elements])

        self.p_table_view("t_sales_model")
        self.p_table_view("t_sales_model_base")

        self.drop_views_if_exists(self.views().df()["table_name"])

        self.con.sql(f"""
            CREATE OR REPLACE VIEW v_demande_quantite AS (
                SELECT
                    HOTEL_CODE,
                    COALESCE(SUM(CASE
                        WHEN {champs} IN ({list_str}) THEN QUANTITE
                        ELSE 0
                    END), 0) AS demande_quantite
                FROM
                    t_sales_model
                GROUP BY
                    hotel_code
                ORDER BY
                    hotel_code
            )
        """)

        self.con.sql("""
            CREATE OR REPLACE VIEW v_part_demande_quantite AS (
                SELECT
                    HOTEL_CODE,
                    produit AS NOM_PRODUIT,
                    produit_nombre_ventes,
                    produit_part_nombre_ventes,
                    demande_quantite,
                    demande_quantite * produit_part_nombre_ventes AS part_demande_quantite
                FROM
                    v_demande_quantite
                LEFT JOIN
                    t_sales_model_base
                USING
                    (HOTEL_CODE)
                ORDER BY
                    HOTEL_CODE,
                    NOM_PRODUIT
            )
        """)

        self.con.sql("""
            CREATE OR REPLACE VIEW v_sales_model_demande AS (
                SELECT 
                    SOLUTION,
                    HOTEL_CODE,
                    HOTEL_NAME,
                    METRES_LINEAIRES,
                    NOM_BOUTIQUE, 
                    TYPE,
                    TYPE_RAW,
                    GAMME,
                    GAMME_RAW,
                    NOM_PRODUIT,
                    NOM_PRODUIT_RAW,
                    CATEGORIE,
                    OPERATEUR,
                    MACHINE,
                    DATE,
                    HEURE,
                    STATUT,
                    CODE_EAN,
                    (QUANTITE / produit_nombre_ventes) * part_demande_quantite AS QUANTITE,
                    (PRIX_HT / produit_nombre_ventes)  * part_demande_quantite AS PRIX_HT,
                    VAT,
                    (PRIX_TTC / produit_nombre_ventes)  * part_demande_quantite AS PRIX_TTC,
                    MARQUE,
                    FOURNISSEUR,
                    - ORDER_ID AS ORDER_ID,
                    TEMPERATURE,
                    (PRIX_TTC_MARCHE / produit_nombre_ventes)  * part_demande_quantite AS PRIX_TTC_MARCHE,
                    (MARGE / produit_nombre_ventes)  * part_demande_quantite AS MARGE
                FROM 
                    t_sales_model
                RIGHT JOIN
                (
                    SELECT
                        *
                    FROM
                        v_part_demande_quantite
                    WHERE
                        part_demande_quantite > 0
                )
                USING
                    (HOTEL_CODE, NOM_PRODUIT)
            )
            """)

        self.con.sql(f"""
            CREATE OR REPLACE VIEW v_sales_model AS (
                SELECT
                    *
                FROM
                    t_sales_model
                WHERE
                    {champs} NOT IN ({list_str})

                UNION ALL

                SELECT
                    *
                FROM
                    v_sales_model_demande
                WHERE
                    {champs} NOT IN ({list_str})
            )
        """)

        # recalcule indicateurs (pipeline) à partir de v_sales_model modifié
        self.p_table_view("v_sales_model_base")


    def remove_produits(self, produits : list[str]):
        self.remove_elements("NOM_PRODUIT", produits)

    def remove_gammes(self, gammes : list[str]):
        self.remove_elements("GAMME", gammes)

    def remove_types(self, types : list[str]):
        self.remove_elements("TYPE", types)


    def _ensure_sim_table_with_scenario_cols(self):
        """
        t_sales_model_sim = base (scénario baseline) + colonnes scénario.
        Recrée la table si les colonnes scénario manquent (anciennes runs).
        """
        self.p_table_view("t_sales_model_base")
        need_rebuild = True
        if self.table_exists("t_sales_model_sim"):
            cols = [
                r[0]
                for r in self.con.sql("DESCRIBE t_sales_model_sim").fetchall()
            ]
            if "scenario_id" in cols and "scenario_label" in cols:
                need_rebuild = False
        if need_rebuild:
            self.con.sql("DROP TABLE IF EXISTS t_sales_model_sim")
            self.con.sql("""
                CREATE TABLE t_sales_model_sim AS
                SELECT
                    'base' AS scenario_id,
                    'base (aucun retrait)' AS scenario_label,
                    'base' AS scenario_kind,
                    0 AS scenario_rank,
                    0 AS n_removed,
                    CAST('' AS VARCHAR) AS removed_items,
                    *
                FROM t_sales_model_base
            """)
            self._scenario_seq = 0
            logging.debug("t_sales_model_sim recréée avec colonnes scénario + baseline")
        else:
            # reprendre le max rank
            try:
                mx = self.con.sql(
                    "SELECT COALESCE(MAX(scenario_rank), 0) FROM t_sales_model_sim"
                ).fetchone()[0]
                self._scenario_seq = int(mx or 0)
            except Exception:
                self._scenario_seq = 0


    def _next_scenario_meta(
        self,
        kind: str,
        elements: list[str],
    ) -> dict:
        self._scenario_seq += 1
        rank = self._scenario_seq
        n = len(elements)
        # label court : kind + n retirés + aperçu
        preview = ", ".join(str(e) for e in elements[:3])
        if n > 3:
            preview += f", … (+{n - 3})"
        sid = f"{kind}_{rank:04d}_n{n}"
        label = f"{kind}: retrait de {n} — {preview}" if n else f"{kind}: baseline"
        return {
            "scenario_id": sid,
            "scenario_label": label,
            "scenario_kind": kind,
            "scenario_rank": rank,
            "n_removed": n,
            "removed_items": " | ".join(str(e) for e in elements),
        }


    def _insert_current_base_as_scenario(self, meta: dict):
        """INSERT le v_sales_model_base courant tagué comme un scénario."""
        # S'assurer que la vue indicateurs est à jour
        self.p_table_view("v_sales_model_base")
        # Échapper quotes pour SQL
        def esc(v):
            return str(v).replace("'", "''")

        self.con.sql(f"""
            INSERT INTO t_sales_model_sim
            SELECT
                '{esc(meta["scenario_id"])}' AS scenario_id,
                '{esc(meta["scenario_label"])}' AS scenario_label,
                '{esc(meta["scenario_kind"])}' AS scenario_kind,
                {int(meta["scenario_rank"])} AS scenario_rank,
                {int(meta["n_removed"])} AS n_removed,
                '{esc(meta["removed_items"])}' AS removed_items,
                *
            FROM v_sales_model_base
        """)
        n = self.con.sql(
            f"SELECT COUNT(*) FROM t_sales_model_sim WHERE scenario_id = '{esc(meta['scenario_id'])}'"
        ).fetchone()[0]
        logging.debug(
            f"INSERT scenario {meta['scenario_id']} → {n} lignes produit "
            f"(rank={meta['scenario_rank']}, n_removed={meta['n_removed']})"
        )


    def multiple_remove_elements(self, champs : str, elements : list[str], kind: str | None = None):
        """
        Pour i = 1..len(elements) : retire les i premiers éléments, calcule
        les indicateurs, INSERT dans t_sales_model_sim avec un scenario_id.
        """
        kind = kind or str(champs).lower()
        self._ensure_sim_table_with_scenario_cols()

        elements = list(elements)
        # i=1 .. len(elements) inclus : retrait cumulatif de 1, 2, …, n éléments
        for i in range(1, len(elements) + 1):
            sub_elements = elements[:i]
            logging.debug(
                f"champs={champs} scenario {i}/{len(elements)} "
                f"removed={len(sub_elements)}"
            )
            self.remove_elements(champs, sub_elements)
            meta = self._next_scenario_meta(kind, sub_elements)
            self._insert_current_base_as_scenario(meta)


    def multiple_remove_produits(self, produits):
        self.multiple_remove_elements("NOM_PRODUIT", list(produits), kind="produit")

    def multiple_remove_gammes(self, gammes):
        self.multiple_remove_elements("GAMME", list(gammes), kind="gamme")

    def multiple_remove_types(self, types):
        self.multiple_remove_elements("TYPE", list(types), kind="type")


    def multiple_remove_produits_gammes_types(self):
        self.p_table_view("v_produits")
        self.p_table_view("v_gammes")
        self.p_table_view("v_types")

        # listes distinctes globales (ordre stable)
        produits = (
            self.con.sql("SELECT DISTINCT produit FROM v_produits ORDER BY produit")
            .df()["produit"]
            .tolist()
        )
        gammes = (
            self.con.sql("SELECT DISTINCT gamme FROM v_gammes ORDER BY gamme")
            .df()["gamme"]
            .tolist()
        )
        types = (
            self.con.sql("SELECT DISTINCT type FROM v_types ORDER BY type")
            .df()["type"]
            .tolist()
        )

        logging.info(
            f"Scénarios prévus : "
            f"produits={len(produits)} + gammes={len(gammes)} + types={len(types)} "
            f"+ 1 baseline → "
            f"~{(len(produits)+len(gammes)+len(types)+1)*7} lignes hôtel×scénario "
            f"(si 7 hôtels)"
        )

        # Table sim = baseline taguée
        self.con.sql("DROP TABLE IF EXISTS t_sales_model_sim")
        self._ensure_sim_table_with_scenario_cols()

        logging.debug("multiple_remove_produits")
        self.multiple_remove_produits(produits)

        logging.debug("multiple_remove_gammes")
        self.multiple_remove_gammes(gammes)

        logging.debug("multiple_remove_types")
        self.multiple_remove_types(types)

        # Vue multi-scénarios pour le pivot
        self.con.sql(
            "CREATE OR REPLACE VIEW v_sales_model_base AS "
            "SELECT * FROM t_sales_model_sim"
        )

        line = self.create_or_replace_sales_model_base_line_table(
            source="t_sales_model_sim"
        )
        n_scen = self.con.sql(
            "SELECT COUNT(DISTINCT scenario_id) FROM t_sales_model_sim"
        ).fetchone()[0]
        n_hotels = self.con.sql(
            "SELECT COUNT(DISTINCT hotel_code) FROM t_sales_model_sim"
        ).fetchone()[0]
        logging.info(
            f"t_sales_model_base_line = {line.shape[0]} lignes × {line.shape[1]} cols "
            f"({n_scen} scénarios × {n_hotels} hôtels attendu ≈ {n_scen * n_hotels})"
        )
        return line


    @classmethod
    def main(cls):
        """Pipeline complet base + simulations multi-scénarios + pivot large."""
        super().main()

        sim = SalesPilSim()
        line = sim.multiple_remove_produits_gammes_types()
        print("line shape (hôtel × scénario) :", line.shape)
        if "scenario_id" in line.columns:
            print(
                "n_scenarios:",
                line["scenario_id"].nunique(),
                "n_hotels:",
                line["hotel_code"].nunique(),
            )
            display(line[["scenario_id", "scenario_label", "hotel_code"]].head(20))
        else:
            display(line.head(3))
        sim.close_con()


In [ ]:
SalesPilSim().main()